# Create file storing gbn shares at the ISCO level (weighted/unweighted by employment shares)
Felix Zaussinger | 02.09.2022

**Core Analysis Goal(s)**
1. Create two sets of shares at the ISCO-level:
    - weighted by KldB employment shares
    - unweighted mean based on ESCO-level classification
        - short_list
        - short_list_tobi
2. Compare results

**Key Insight(s)**
1.
2.
3.

In [86]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from src import utils
import mapping_career_causeways

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload

# OPTIONAL: always reload modules so that as you change code in src, it gets loaded
%autoreload 2

# Settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sns.set_context("poster")
sns.set(rc={"figure.figsize": (16, 9.0)})
sns.set_style("ticks")

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# load paths
useful_paths = utils.UsefulPaths()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


#### Prepare unweighted shares at all ISCO-08 levels
- for both short_list and short_list_tobi

Read final GBN classifications (ESCO-level)

In [87]:
df_sl = pd.read_csv(
    os.path.join(
        useful_paths.data_processed,
        "esco",
        "esco_level_gbn_classification_short_lists.csv",
    ),
    index_col=0,
)

df_sl_tobi = pd.read_csv(
    os.path.join(
        useful_paths.data_processed,
        "esco",
        "esco_level_gbn_classification_short_lists_tobi.csv",
    ),
    index_col=0,
)

Create ESCO-ISCO correspondance file

In [88]:
from src.data.framework import Esco

esco = Esco()
occ = esco.occupations
isco_col = "isco_code"
data_container = []

In [89]:
for n_digits_isco in [1, 2, 3, 4]:
    # Create ESCO-ISCO correspondance file
    occ[isco_col] = occ.iscoGroup.str[:n_digits_isco]
    esco_to_isco = pd.merge(
        occ,
        esco.isco_groups[["code", "preferredLabel"]],
        left_on=isco_col,
        right_on="code",
        how="left",
        suffixes=["_esco", "_isco"],
    )
    esco_to_isco = esco_to_isco[["conceptUri", isco_col, "preferredLabel_isco"]]

    # Merge ISCO codes, calculate per-category count of ESCO occupations within ISCO groups
    df_sl_merged = df_sl.merge(esco_to_isco, on="conceptUri", how="left")
    df_sl_tobi_merged = df_sl_tobi.merge(esco_to_isco, on="conceptUri", how="left")

    df_sl_merged["n_esco"] = 1
    df_sl_merged_by_isco = df_sl_merged.groupby(
        [isco_col, "preferredLabel_isco", "gbn_classification_short_list"]
    )["n_esco"].count()
    df_sl_merged_by_isco = df_sl_merged_by_isco.reset_index()
    df_sl_merged_by_isco["isco_level"] = n_digits_isco

    df_sl_tobi_merged["n_esco"] = 1
    df_sl_tobi_merged_by_isco = df_sl_tobi_merged.groupby(
        [isco_col, "preferredLabel_isco", "gbn_classification_short_list"]
    )["n_esco"].count()
    df_sl_tobi_merged_by_isco = df_sl_tobi_merged_by_isco.reset_index()
    df_sl_tobi_merged_by_isco["isco_level"] = n_digits_isco

    # Calc unweighted shares at ISCO-08 3-digit level: df_sl
    cnt = 0
    data_store = []
    cols_all = {"green", "brown", "neutral"}

    for grp, grp_df in df_sl_merged_by_isco.groupby(isco_col):

        # pivot long to wide, simplify multi-indices
        grp_df_piv = grp_df.pivot(
            index=["isco_level", isco_col, "preferredLabel_isco"],
            columns=["gbn_classification_short_list"],
            values=["n_esco"],
        )
        grp_df_piv.columns = grp_df_piv.columns.get_level_values(1)
        grp_df_piv = grp_df_piv.reset_index()
        # grp_df_piv.index = grp_df_piv.index.get_level_values(0)

        # create missing column
        cols_exist = set(grp_df_piv.columns.tolist())
        col_missing = list(cols_all - cols_exist)
        grp_df_piv[col_missing] = 0

        # calc shares
        grp_df_piv["N"] = grp_df_piv[list(cols_all)].sum(axis=1)

        for col in list(cols_all):
            grp_df_piv["share_{}".format(col)] = grp_df_piv[col] / grp_df_piv["N"]

        # append
        data_store.append(grp_df_piv)

        # concat to new df
        df_out = pd.concat(data_store)

        # save individual file
        df_out.to_csv(
            os.path.join(
                useful_paths.data_processed,
                "esco",
                "short_list_gbn_shares_by_isco{}d_unweighted.csv".format(n_digits_isco),
            )
        )

        # flag list version
        df_out["list_version"] = "short_list"

        # global append
        data_container.append(df_out)

    # Calc unweighted shares at ISCO-08 3-digit level: df_sl_tobi
    cnt = 0
    data_store = []
    cols_all = {"green", "brown", "neutral"}

    for grp, grp_df in df_sl_tobi_merged_by_isco.groupby(isco_col):

        # pivot long to wide, simplify multi-indices
        grp_df_piv = grp_df.pivot(
            index=["isco_level", isco_col, "preferredLabel_isco"],
            columns=["gbn_classification_short_list"],
            values=["n_esco"],
        )
        grp_df_piv.columns = grp_df_piv.columns.get_level_values(1)
        grp_df_piv = grp_df_piv.reset_index()
        # grp_df_piv.index = grp_df_piv.index.get_level_values(0)

        # create missing column
        cols_exist = set(grp_df_piv.columns.tolist())
        col_missing = list(cols_all - cols_exist)
        grp_df_piv[col_missing] = 0

        # calc shares
        grp_df_piv["N"] = grp_df_piv[list(cols_all)].sum(axis=1)

        for col in list(cols_all):
            grp_df_piv["share_{}".format(col)] = grp_df_piv[col] / grp_df_piv["N"]

        # append
        data_store.append(grp_df_piv)

    # concat to new df
    df_out = pd.concat(data_store)

    # save individual file
    df_out.to_csv(
        os.path.join(
            useful_paths.data_processed,
            "esco",
            "short_list_tobi_gbn_shares_by_isco{}d_unweighted.csv".format(
                n_digits_isco
            ),
        )
    )

    # flag list version
    df_out["list_version"] = "short_list_tobi"

    # global append
    data_container.append(df_out)

In [135]:
df_sl_tobi_merged_by_isco

,isco_code,preferredLabel_isco,gbn_classification_short_list,n_esco,isco_level
0,0110,Commissioned armed forces officers,neutral,12,4
1,0210,Non-commissioned armed forces officers,neutral,4,4
2,0310,"Armed forces occupations, other ranks",neutral,5,4
3,1111,Legislators,neutral,7,4
4,1112,Senior government officials,neutral,9,4
...,...,...,...,...,...
477,9613,Sweepers and related labourers,neutral,1,4
478,9621,"Messengers, package deliverers and luggage por...",neutral,2,4
479,9622,Odd job persons,neutral,1,4
480,9623,Meter readers and vending-machine collectors,neutral,2,4


Combine to single long DF and save (more compact than single csvs)

In [90]:
gbn_shares_final = pd.concat(data_container).drop_duplicates()
gbn_shares_final = gbn_shares_final.sort_values(
    ["list_version", "isco_level"], ascending=True
)
gbn_shares_final = gbn_shares_final.reset_index(drop=True)

utils.save_df_to_files(
    df=gbn_shares_final,
    output_dir=os.path.join(useful_paths.data_processed, "esco"),
    fname_no_ext="final_gbn_shares_by_isco_unweighted",
)

In [91]:
gbn_shares_final

gbn_classification_short_list,isco_level,isco_code,preferredLabel_isco,neutral,green,brown,N,share_green,share_brown,share_neutral,list_version
0,1,0,Armed forces occupations,21,0,0,21,0.000000,0.000000,1.000000,short_list
1,1,1,Managers,331,3,4,338,0.008876,0.011834,0.979290,short_list
2,1,2,Professionals,782,39,25,846,0.046099,0.029551,0.924350,short_list
3,1,3,Technicians and associate professionals,605,36,12,653,0.055130,0.018377,0.926493,short_list
4,1,4,Clerical support workers,87,0,2,89,0.000000,0.022472,0.977528,short_list
...,...,...,...,...,...,...,...,...,...,...,...
1201,4,9613,Sweepers and related labourers,1,0,0,1,0.000000,0.000000,1.000000,short_list_tobi
1202,4,9621,"Messengers, package deliverers and luggage por...",2,0,0,2,0.000000,0.000000,1.000000,short_list_tobi
1203,4,9622,Odd job persons,1,0,0,1,0.000000,0.000000,1.000000,short_list_tobi
1204,4,9623,Meter readers and vending-machine collectors,2,0,0,2,0.000000,0.000000,1.000000,short_list_tobi


#### Prepare weighted shares (via KldB weights)

ISCO-08 3-digit level

In [92]:
gbn_shares_wtd_3d_raw = pd.read_csv(
    os.path.join(
        useful_paths.project_dir,
        "rcode",
        "D_Ergebnisse",
        "_OUTPUT_FINAL_GEPRUEFT",
        "weighted_occ_shares_ISCO3D_geprueft_formatiert.csv",
    ),
    dtype={"EF541UG1": str},
)

# join isco names
gbn_shares_wtd_3d = (
    gbn_shares_wtd_3d_raw.merge(
        right=esco.isco_groups[["conceptUri", "preferredLabel", "code"]],
        left_on="EF541UG1",
        right_on="code",
        how="left",
    )
    .rename(columns={"code": "isco_code"})
    .drop(columns=["EF541UG1", "conceptUri"])
)

col_order = [
    "isco_code",
    "preferredLabel",
    "n_obs",
    "EF952_sum",
    "share_brown_unwtd",
    "share_brown_wtd",
    "share_green_unwtd",
    "share_green_wtd",
    "share_neutral_unwtd",
    "share_neutral_wtd",
]

gbn_shares_wtd_3d = gbn_shares_wtd_3d[col_order]
gbn_shares_wtd_3d.to_csv(
    os.path.join(
        useful_paths.project_dir,
        "rcode",
        "D_Ergebnisse",
        "_OUTPUT_FINAL_GEPRUEFT",
        "weighted_occ_shares_ISCO3D_named.csv",
    )
)

In [93]:
gbn_shares_wtd_3d

,isco_code,preferredLabel,n_obs,EF952_sum,share_brown_unwtd,share_brown_wtd,share_green_unwtd,share_green_wtd,share_neutral_unwtd,share_neutral_wtd
0,011,Commissioned armed forces officers,105.0,12347.157,0.0,0.0,0.000000,0.000000,1.000000,1.000000
1,021,Non-commissioned armed forces officers,72.0,9550.961,0.0,0.0,0.000000,0.000000,1.000000,1.000000
2,031,"Armed forces occupations, other ranks",1263.0,168173.179,0.0,0.0,0.000000,0.000000,1.000000,1.000000
3,111,Legislators and senior officials,383.0,40370.029,0.0,0.0,0.000000,0.000000,1.000000,1.000000
4,112,Managing directors and chief executives,6730.0,734645.324,0.0,0.0,0.000000,0.000000,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...
118,932,Manufacturing labourers,3351.0,383081.002,0.0,0.0,0.000000,0.000000,1.000000,1.000000
119,933,Transport and storage labourers,6395.0,742086.441,0.0,0.0,0.000000,0.000000,1.000000,1.000000
120,941,Food preparation assistants,3264.0,385568.918,0.0,0.0,0.000000,0.000000,1.000000,1.000000
121,961,Refuse workers,510.0,57909.302,0.0,0.0,0.673856,0.673845,0.326144,0.326155


ISCO-08 4-digit level

In [94]:
gbn_shares_wtd_4d_raw = pd.read_csv(
    os.path.join(
        useful_paths.project_dir,
        "rcode",
        "D_Ergebnisse",
        "_OUTPUT_FINAL_GEPRUEFT",
        "weighted_occ_shares_ISCO4D_bearbeitet.csv",
    ),
    dtype={"EF541": str},
)

# join isco names
gbn_shares_wtd_4d = (
    gbn_shares_wtd_4d_raw.merge(
        right=esco.isco_groups[["conceptUri", "preferredLabel", "code"]],
        left_on="EF541",
        right_on="code",
        how="left",
    )
    .rename(columns={"code": "isco_code"})
    .drop(columns=["EF541", "conceptUri"])
)

col_order = [
    "isco_code",
    "preferredLabel",
    "n_obs",
    "EF952_sum",
    "share_brown_unwtd",
    "share_brown_wtd",
    "share_green_unwtd",
    "share_green_wtd",
    "share_neutral_unwtd",
    "share_neutral_wtd",
]

gbn_shares_wtd_4d = gbn_shares_wtd_4d[col_order]
gbn_shares_wtd_4d.to_csv(
    os.path.join(
        useful_paths.project_dir,
        "rcode",
        "D_Ergebnisse",
        "_OUTPUT_FINAL_GEPRUEFT",
        "weighted_occ_shares_ISCO4D_named.csv",
    )
)

In [95]:
gbn_shares_wtd_4d

,isco_code,preferredLabel,n_obs,EF952_sum,share_brown_unwtd,share_brown_wtd,share_green_unwtd,share_green_wtd,share_neutral_unwtd,share_neutral_wtd
0,0110,Commissioned armed forces officers,105.0,12347.157,0.0,0.0,0.000000,0.000000,1.000000,1.000000
1,0210,Non-commissioned armed forces officers,72.0,9550.961,0.0,0.0,0.000000,0.000000,1.000000,1.000000
2,0310,"Armed forces occupations, other ranks",1263.0,168173.179,0.0,0.0,0.000000,0.000000,1.000000,1.000000
3,1111,Legislators,113.0,12220.074,0.0,0.0,0.000000,0.000000,1.000000,1.000000
4,1112,Senior government officials,211.0,22031.293,0.0,0.0,0.000000,0.000000,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...
385,9612,Refuse sorters,143.0,16.212.215,0.0,0.0,0.692308,0.692308,0.307692,0.307692
386,9613,Sweepers and related labourers,367.0,41.697.087,0.0,0.0,0.666667,0.666667,0.333333,0.333333
387,9621,"Messengers, package deliverers and luggage por...",1572.0,172.449.721,0.0,0.0,0.000000,0.000000,1.000000,1.000000
388,9623,Meter readers and vending-machine collectors,24.0,2.572.723,0.0,0.0,0.000000,0.000000,1.000000,1.000000


#### Comparison with unweighted shares sorely from ESCO (w/o KldB crosswalk)

In [126]:
# rename certain cols of unweighted ESCO ds
ren = {
    "N": "n_esco",
    "brown": "n_esco_brown",
    "green": "n_esco_green",
    "neutral": "n_esco_neutral",
    "share_brown": "share_brown_esco",
    "share_green": "share_green_esco",
    "share_neutral": "share_neutral_esco",
}
gbn_shares_final = gbn_shares_final.rename(columns=ren)

col_order_output = [
    "isco_level",
    "isco_code",
    "preferredLabel",
    "list_version",
    "n_esco",
    "n_esco_brown",
    "n_esco_green",
    "n_esco_neutral",
    "n_obs",
    "EF952_sum",
    "share_brown_esco",
    "share_brown_unwtd",
    "share_brown_wtd",
    "share_green_esco",
    "share_green_unwtd",
    "share_green_wtd",
    "share_neutral_esco",
    "share_neutral_unwtd",
    "share_neutral_wtd",
]

3 Digits

In [131]:
import pingouin as pg

gbn_shares_comp_3d = gbn_shares_wtd_3d.merge(
    gbn_shares_final, on="isco_code", how="left"
).drop(columns=["preferredLabel_isco"])
gbn_shares_comp_3d = gbn_shares_comp_3d[col_order_output]
gbn_shares_comp_sl_3d = gbn_shares_comp_3d.query("list_version == 'short_list'")
gbn_shares_comp_slt_3d = gbn_shares_comp_3d.query("list_version == 'short_list_tobi'")

In [132]:
gbn_shares_comp_3d.to_csv(
    os.path.join(
        useful_paths.data_processed, "esco", "final_gbn_shares_by_isco_3d_weighted.csv"
    )
)

In [129]:
gbn_shares_comp_sl_3d.iloc[
    :, gbn_shares_comp_sl_3d.columns.str.startswith("share")
].rcorr()

,share_brown_unwtd,share_brown_wtd,share_green_unwtd,share_green_wtd,share_neutral_unwtd,share_neutral_wtd,share_green_esco,share_brown_esco,share_neutral_esco
share_brown_unwtd,-,***,,,***,***,,***,***
share_brown_wtd,1.0,-,,,***,***,,***,***
share_green_unwtd,0.088,0.09,-,***,***,***,***,,**
share_green_wtd,0.088,0.09,1.0,-,***,***,***,,**
share_neutral_unwtd,-0.482,-0.483,-0.915,-0.915,-,***,***,**,***
share_neutral_wtd,-0.479,-0.48,-0.917,-0.917,1.0,-,***,**,***
share_green_esco,0.051,0.052,0.464,0.465,-0.429,-0.431,-,,***
share_brown_esco,0.624,0.626,-0.003,-0.003,-0.249,-0.248,-0.022,-,***
share_neutral_esco,-0.539,-0.541,-0.276,-0.277,0.46,0.46,-0.582,-0.8,-


In [99]:
gbn_shares_comp_slt_3d.iloc[
    :, gbn_shares_comp_slt_3d.columns.str.startswith("share")
].rcorr()

,share_brown_unwtd,share_brown_wtd,share_green_unwtd,share_green_wtd,share_neutral_unwtd,share_neutral_wtd,share_green,share_brown,share_neutral
share_brown_unwtd,-,***,,,***,***,,***,***
share_brown_wtd,1.0,-,,,***,***,,***,***
share_green_unwtd,0.088,0.09,-,***,***,***,***,,***
share_green_wtd,0.088,0.09,1.0,-,***,***,***,,***
share_neutral_unwtd,-0.482,-0.483,-0.915,-0.915,-,***,***,***,***
share_neutral_wtd,-0.479,-0.48,-0.917,-0.917,1.0,-,***,***,***
share_green,0.051,0.052,0.464,0.465,-0.429,-0.431,-,,***
share_brown,0.929,0.929,0.101,0.101,-0.464,-0.462,0.064,-,***
share_neutral,-0.431,-0.431,-0.452,-0.453,0.572,0.572,-0.91,-0.471,-


4 digits

In [133]:
gbn_shares_comp_4d = gbn_shares_wtd_4d.merge(
    gbn_shares_final, on="isco_code", how="left", suffixes=("", "_esco")
).drop(columns=["preferredLabel_isco"])
gbn_shares_comp_4d = gbn_shares_comp_4d[col_order_output]
gbn_shares_comp_sl_4d = gbn_shares_comp_4d.query("list_version == 'short_list'")
gbn_shares_comp_slt_4d = gbn_shares_comp_4d.query("list_version == 'short_list_tobi'")

In [134]:
gbn_shares_comp_4d.to_csv(
    os.path.join(
        useful_paths.data_processed, "esco", "final_gbn_shares_by_isco_4d_weighted.csv"
    )
)

In [101]:
gbn_shares_comp_sl_4d.iloc[
    :, gbn_shares_comp_sl_4d.columns.str.startswith("share")
].rcorr()

,share_brown_unwtd,share_brown_wtd,share_green_unwtd,share_green_wtd,share_neutral_unwtd,share_neutral_wtd,share_green,share_brown,share_neutral
share_brown_unwtd,-,***,,,***,***,,***,***
share_brown_wtd,1.0,-,,,***,***,,,
share_green_unwtd,0.018,0.018,-,***,***,***,***,,***
share_green_wtd,0.018,0.015,1.0,-,***,***,***,,***
share_neutral_unwtd,-0.663,-0.663,-0.76,-0.76,-,***,***,***,***
share_neutral_wtd,-0.663,-0.663,-0.76,-0.76,1.0,-,***,***,***
share_green,0.027,0.069,0.677,0.675,-0.525,-0.525,-,,***
share_brown,0.434,0.046,-0.029,-0.03,-0.26,-0.261,-0.034,-,***
share_neutral,-0.392,-0.076,-0.324,-0.324,0.497,0.498,-0.468,-0.867,-


In [102]:
gbn_shares_comp_slt_4d.iloc[
    :, gbn_shares_comp_slt_4d.columns.str.startswith("share")
].rcorr()

,share_brown_unwtd,share_brown_wtd,share_green_unwtd,share_green_wtd,share_neutral_unwtd,share_neutral_wtd,share_green,share_brown,share_neutral
share_brown_unwtd,-,***,,,***,***,,***,***
share_brown_wtd,1.0,-,,,***,***,,*,*
share_green_unwtd,0.018,0.018,-,***,***,***,***,,***
share_green_wtd,0.018,0.015,1.0,-,***,***,***,,***
share_neutral_unwtd,-0.663,-0.663,-0.76,-0.76,-,***,***,***,***
share_neutral_wtd,-0.663,-0.663,-0.76,-0.76,1.0,-,***,***,***
share_green,0.027,0.069,0.677,0.675,-0.525,-0.525,-,,***
share_brown,0.769,0.105,-0.002,-0.002,-0.498,-0.499,0.008,-,***
share_neutral,-0.518,-0.12,-0.509,-0.508,0.718,0.718,-0.728,-0.691,-
